In [1]:
from datasets import load_dataset, Dataset, Image as Img
import glob
import os
import cv2
from IPython.display import Image, clear_output
import matplotlib.pyplot as plt
import torch
from cairosvg import svg2png
from transformers import AutoProcessor
import PIL
import numpy as np

## Model settings, toknizer

In [2]:
checkpoint = "microsoft/git-base"

In [3]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(checkpoint)

/home/lanv/venv/lib64/python3.12/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/lanv/venv/lib64/python3.12/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:
model.load_state_dict(torch.load("svgmodel_16.pth"))

<All keys matched successfully>

In [5]:
from transformers import AutoProcessor, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
processor = AutoProcessor.from_pretrained(checkpoint)
    
def tokenize(batch):
    return tokenizer(batch, padding=True, truncation=True, max_length=5000)

In [6]:
icons = "clean_icons"
images = "clean_images"

svgs = []
pngs = []

ma = 10000000

for f in glob.glob(os.path.join("data", icons,"*.svg")):
  img = (f.split("/")[-1].split(".")[0])

  if(len(open(f).read()) < 500):
    svgs.append(os.path.join("data", icons, img+".svg"))
    pngs.append(os.path.join("data", images, img+".png"))
  
    ma -= 1
    if(ma == 0):
      break

In [7]:
def path_to_encoded(d_in):
  text = open(d_in).read()
  return {"labels": text}

def to_rgb(im_in):
  return {"pixel_values":im_in.convert("RGB")}

ds = (
    Dataset
    .from_dict({
    "svg": svgs,
    "png": pngs
    })
    .cast_column("png", Img())
    .rename_column("png","pixel_values")
    .rename_column("svg","labels")
    .map(path_to_encoded, desc="read and tokenize SVG", input_columns="labels")
    .map(to_rgb, desc="convert images", input_columns="pixel_values")
  )
ds.set_format("torch")
ds

read and tokenize SVG:   0%|          | 0/1996 [00:00<?, ? examples/s]

convert images:   0%|          | 0/1996 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'pixel_values'],
    num_rows: 1996
})

In [8]:

processor = AutoProcessor.from_pretrained("microsoft/git-base")

class ImageCaptioningDataset(Dataset):
    def __init__(self, dataset, processor):
        self.dataset = dataset
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]

        encoding = self.processor(images=item["pixel_values"], text=item["labels"], padding="max_length", return_tensors="pt")

        # remove batch dimension
        encoding = {k:v.squeeze() for k,v in encoding.items()}

        return encoding

train_dataset = ImageCaptioningDataset(ds, processor)

In [9]:
print(ds[0].keys())
ds[0]["pixel_values"].shape

dict_keys(['labels', 'pixel_values'])


torch.Size([3, 256, 256])

In [10]:
from torch.utils.data import DataLoader
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=8)

In [11]:
def renderSvg(svg):
  try:
    return torch.tensor(PIL.image.fromBytes(svg2png(svg)))
  except:
    return torch.zeros((3, 224, 224))

In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [13]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6)
model.to(device)
criteron = torch.nn.MSELoss()
model.train()
clear_output()

In [14]:
losses = []

for epoch in range(50):
  for idx, batch in enumerate(train_dataloader):
    input_ids = batch.pop("input_ids").to(device)
    pixel_values = batch.pop("pixel_values").to(device)

    outputs = model(input_ids=input_ids,
                    pixel_values=pixel_values,
                    labels=input_ids)
    
    print("shp", pixel_values.shape)
    print("outs", outputs["logits"])
    generated_ids = model.generate(pixel_values=pixel_values, max_length=50)
    generated_captions = processor.batch_decode(generated_ids, skip_special_tokens=True)
    print("cappts", generated_captions)

    rendered = []
    for o in generated_captions:
      print(o)
      rendered.append(renderSvg(o))
    
    rendered = np.array(rendered)
    print("shp2", rendered.shape)
    
    rendered_pred = torch.from_numpy(rendered).to(device)
    loss_img = criteron(rendered_pred, pixel_values)

    loss = outputs.loss  * loss_img

    # graphing
    l = loss.cpu().item()
    losses.append(l)
    clear_output()
    ax = plt.subplot()
    ax.plot(losses)
    ax.set_yscale("log")
    plt.show()
    print(f"{epoch} Loss: {l}")
  
    # updating
    loss.backward()

    optimizer.step()
    optimizer.zero_grad()

Unused or unrecognized kwargs: padding.
We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


/home/lanv/venv/lib64/python3.12/site-packages/torch/nn/modules/conv.py:456: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:84.)
  return F.conv2d(input, weight, bias, self.stride,


shp torch.Size([8, 3, 224, 224])
outs tensor([[[ 3.6062, -2.5186, -2.5191,  ..., -2.5178, -2.5191, -2.5171],
         [ 3.6173, -2.6753, -2.6768,  ..., -2.6740, -2.6765, -2.6727],
         [ 3.4718, -2.4352, -2.4360,  ..., -2.4343, -2.4359, -2.4335],
         ...,
         [12.2500, -6.3525, -6.3568,  ..., -6.3497, -6.3560, -6.3458],
         [12.2553, -6.3595, -6.3635,  ..., -6.3570, -6.3628, -6.3534],
         [12.2144, -6.3767, -6.3812,  ..., -6.3739, -6.3803, -6.3699]],

        [[ 3.7259, -2.4727, -2.4738,  ..., -2.4719, -2.4735, -2.4708],
         [ 3.5400, -2.3500, -2.3501,  ..., -2.3497, -2.3502, -2.3494],
         [ 3.8403, -2.6105, -2.6110,  ..., -2.6100, -2.6111, -2.6094],
         ...,
         [12.3658, -6.2248, -6.2281,  ..., -6.2227, -6.2277, -6.2196],
         [12.3462, -6.2737, -6.2777,  ..., -6.2712, -6.2770, -6.2676],
         [12.2297, -6.3143, -6.3185,  ..., -6.3117, -6.3177, -6.3079]],

        [[ 3.9752, -2.7292, -2.7299,  ..., -2.7287, -2.7298, -2.7279],
       

Unused or unrecognized kwargs: padding.


cappts ['< svg xmlns = " http : / / www. w3. org / 2000 / svg " width = " 256 " height = " 256 " viewbox = " 0 0 256 256 " fill = " none " stroke =', '< svg xmlns = " http : / / www. w3. org / 2000 / svg " width = " 256 " height = " 256 " viewbox = " 0 0 256 256 " fill = " none " stroke =', '< svg xmlns = " http : / / www. w3. org / 2000 / svg " width = " 256 " height = " 256 " viewbox = " 0 0 256 256 " fill = " none " stroke =', '< svg xmlns = " http : / / www. w3. org / 2000 / svg " width = " 256 " height = " 256 " viewbox = " 0 0 256 256 " fill = " none " stroke =', '< svg xmlns = " http : / / www. w3. org / 2000 / svg " width = " 256 " height = " 256 " viewbox = " 0 0 256 256 " fill = " none " stroke =', '< svg xmlns = " http : / / www. w3. org / 2000 / svg " width = " 256 " height = " 256 " viewbox = " 0 0 256 256 " fill = " none " stroke =', '< svg xmlns = " http : / / www. w3. org / 2000 / svg " width = " 256 " height = " 256 " viewbox = " 0 0 256 256 " fill = " none " stroke ='

OutOfMemoryError: CUDA out of memory. Tried to allocate 186.00 MiB. GPU 